# 📋 STR01 · 미션 02 | 부대별 일일 소모율 정규화 및 재고 변동 데이터 전처리 모듈 구현

**학생용 Student · 직접 실습**  
CMS 편성: 2시간 · 교육용 가상 데이터 사용

---

## 📖 1. 상황과 이번 미션의 문제

가상의 20개 조직과 5종 물품의 기록을 이용해 **부대별 일일 소모율 정규화 및 재고 변동 데이터 전처리 모듈 구현** 과제를 수행합니다.

실제 조직의 기록이 아닌 실습자료이며, 코드를 실행한 결과도 실제 현장 상태를 뜻하지 않습니다.

**이번 미션에서 해결할 문제:** 중복, 누락 일자, 소모량 결측, 단위 혼용, 음수 재고, 품목명 오타, 태그 없는 이상 급증을 일부러 섞은 가상 원장을 만들어 정제 규칙 10종을 적용한다.

긴급 불출과 비계획 정비로 태그된 급증은 이상치로 깎지 않고 보존한다.

정제 후 정합성 검증 10항목의 통과율과 원본 복원 품질을 정제 품질 보고서로 출력한다.

**만들 결과물:** `src/etl/str01_data_cleansing.py`, `reports/str01_cleansing_report.md`, `outputs/M02/df.csv`, `forms/STR01_M02_forms.md`

## 📌 2. 처음 시작하는 방법

1. 아래 **환경 준비 셀**을 실행합니다. Colab에서는 메뉴의 **런타임 → 모두 실행** 대신 각 셀의 결과를 먼저 확인하면서 진행해도 됩니다.
2. 핵심 함수의 `TODO`를 AI와 함께 구현합니다. 처음에는 함수의 입력·출력과 왜 필요한지 질문하세요.
3. 첫 실행 결과를 확인하고 실패하면 오류 메시지를 AI에게 설명해 달라고 요청합니다.
4. AI가 제안한 코드 중 최소 3가지를 원자료와 실행 결과로 대조하고 수정 이유를 기록합니다.
5. 활동별 작성 칸을 채운 뒤 **실제 AI 대화 원문**을 보존하고 마지막 저장 셀로 ZIP을 만듭니다.

> **보안과 평가:** 실제 조직·위치·재고·작전 자료, 개인정보를 외부 AI에 입력하지 않습니다.

- 모델 성능은 **실제 출력 로그**로 판단하며, 예시값을 실측으로 적지 않습니다.
- 실행 성공이나 ZIP 저장은 학습 PASS·실제 승인과 다릅니다.
- 폐쇄망에서는 허용된 Python/Jupyter 환경에서 실행하세요.
- 인터넷이 필요한 `pip` 설치는 폐쇄망에서 실패할 수 있으므로 사전 패키지 준비가 필요합니다.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 업로드 파일명은 자유이며, .zip 확장자와 유효한 ZIP 내용만 확인합니다.
current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'STR01_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
STR01_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global STR01_PROJECT_READY, STR01_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    STR01_PROJECT_DIR = WORKSPACE
    STR01_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 2개: STR01_M02_forms.md, STR01_M02_lesson.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAAIAAAAIVxuE3ryjwgAAIAUAAASAAAAU1RSMDFfTTAyX2Zvcm1zLm1k5VjdTxtXFn/3X3GlaN/4sEP6sUirqo+V2j5k921VIZqQVaQEIkxatUIr'
    'A2PkYFcxiQdsMrbsFgKkjjLgMR5Usivxp6Rvc+/8D/s75947HhuUvGyfqqDA/T4fv/M758wN8fd/3M7mxFfZm0K1tpVzItTOtio3xbtCTcizgqwUZM8RqnmB'
    'H6E2K/LVkfKOhGq70cCLG9jkP8XRbtRrC9kryKcNIX/yVTOIHR+7HHValy+7Asfk85KI+t247mQy6kVV9kIRP30S+YVZ8c/YdVSp/g1dq9qewNXqvHE5iLfO'
    'MS1UvaQaIa4PVCcQWrTUG6kjIpdVPxdVExK319TecezWSTDsid1j0i46LaiXTRFv9qPeBSmSy2b/gpvksT8hojCIwidQu6TOvMuBPHeinhPv/UTnMRCR31DN'
    'ApTwSaoR4aCGUOuvIYms1tUhBu6WNgnJjDm5VcOeqUwGRopO3wp6vu3NitsfCbWxFrseDEnXyoMLMaN2q8rxxE3Zoela9F8o3OvDaHhtX/lHqlEYfV7cykYn'
    'gWrWIeMz8yhZgVToV6G7LJdkeX9KaMPzEm8yBpLbuLq8H+88oTuMSbEDaxCEZJI1n6RR7YJqHQwvzGRu3NBGH3VEJrNqXbIqLAYqBah6OYgGF9GJP5wnyaHJ'
    '6qifLwckazcUynPUOZZx4+TkpLjmf6zczmFgf2h8c2w8Mza+NTb+aGz88dj4k7Hxp2Pjv46Nc9n0BFnJ2AcxEb+wniEzabuT+mkcYKxBKmL3tXx1PM2xFNC2'
    '5oX0AT+cGQHF0IDAAGJnw4sGoQX8e6yX2DAKfbVfSImtVYEwEH1smhT6/As6gviAS31P+ggZ8xqmZaeZyUwiJroQXMSVqiwfJRRT9lS7Oive7XToElnuIygY'
    'biFzBi+YWbUbyFZFkMn0aVz6+RcIECEPO1EIeOwWZTegiEAYK689S6+2vajXj11Efqsqf6kAS29r8jkExIpy35JZYTiAH2EJE/mYoHMcGvgJ2JLaHXqV7wdJ'
    'Ha4xlR0EdBDKwTmz5ERgm8nw1TGIR7BT2dl7NXX6Whgky3qVj3EMJbi2e63C791rJawLeVyDJB8Ii1FPjjhQiwaW42eJ16zMLECaIobswVYUhsKMiNbnbCem'
    'Cr+u6hrKuKDhGCvIp44BRpo+VgkBytmXr4qJzvoV5bTjjSbZx/qQAwXILlbU1n68BXBbJhmaRSONHcxwoPNeG4yM8wxkaNwL5AEbvu8RcWl5cbKC0MK2D/LN'
    'dea9YuFrJzIzIDuWbGONLQxK7PhpC7OuQp6W4npIxlT+f7SRNwjziGjIRvlFHlaJuGM3kOVzIdcDZDvKY+t160fkLrXuW7Ol/EOpcauf8oD2J4Md/tk6gIkr'
    'FQmaeRNShrUbODRtIMlTh4bJGgfo+JrhuhEJUrjX5jd+tu76P5id2VZXGJrT+FnzGgjgzCOtdfFh6FVY+mPulGcOYgt0FvqR7wxDVGuQymCy3AUl4cCbEH+S'
    '3arElagg6OpVGwB/CJauqj3JSgalxBFmSBEWlJRbsiWUdgEzpXVZMURAybM1Dg+d9knPUL6Evsj8lEu07VKnASDVBFrfhKpYV94F1ytOnWgZ+nOwDckV6S4K'
    'CkQX8Wabk92Gp3MAooCYn/9MMwbPKNQ5zfD3Qk2WCeoIEOiFcrTsWT/8XnDZ5bHbV+WuqTGTHARHu0emvmgVZRvohhL4LeKGi5u5DvlVWHIx2S4F2yH3+l50'
    '2qUwQcIFgySbLRzeS8WLS8sP5x/c/3Fh7s7SYv7xw0cr95cWsc7/UBa+CY3g9GzliInQOUE0DoHJicl1ZLmR0AVJ7pwg2RlpGelnVGqOBKYp7a4WhNpwDQ91'
    'O9f5hw5bYSyPr5otVJRooyW1go0MNsGQPI1/GU0fiACsUPFGLYgut2ff03jo0t70Hra6v7770FRHNZfJ8+PdCN2lbU4buAG4prKnkzYRemQy4mHqV/ZqDAWD'
    'KU0VlE4hJn5T4bN3TGFALtQVDznnWhggceEeA0vibfctK8pp0ZboNsXpkkrnucsBaiGYvOVYn2kTsmo61+6UdOvxmUHaqrhprc26z4r88p3phZUH0/mV5Wxu'
    '7u78yvzcnQcL84v5+4v/mnr0A2XVa6WGgW07QBJquRGooF6qWe4tzz9cmPj+/uLdpe//lrsl3m0+E9/O5xcuB8vzKwvsHQaqLhT6UAsqThnfg0qflcDLhAG5'
    'jvztqZLHcNcXDkEh1P62chsUHpqAUvDwUj0PXKKrY10Yc6GSboE0orNUCH09/zWlUj6LEu+Fk/Q2vEe3i4z8skf5Ia5dELDo7ee+af5gdLoA3ARfqLNj1Tmx'
    'MDz9VYespRzNrtB425vWjIsjRrah9wjUL5wvF/J5jRe2nY5xeoHDULM5AGpKsKu+n7G+Z7D8mTpvw1RJnJoUj/ILJZmm8nPyoU7hVnxOoNOa5xMWNSQ6tK4x'
    'we3cv9H3saA+tbrcmO0CqL9cUHizbRwSFEhJ1XyEKkraTGOaqKh9kVXGMupzUxWarwJq5zeuCPtV5XWII0g0sJBNCUkVjyDkN/EXGNsQt9rdpDRCGf7cmmtY'
    'DXITRubR3aapf4YRRl1COiJsKMATYcdwIX9BaIaciZhuqXfC5fJlIiO5nnTIZdE4jmFkvWsx0gzYj8TV8ESxwBcfEtRt40v3sP9HkqLtfPmzAjduIIDWEC/P'
    'qBO58mWEiiTy08G0Lkhn+IOKoxMbhSjXLTahwUKtrgbENVF2y0QZNsJvH2TYCbG88GhpeSVvNiRrc3p+6uHdCbH0eOXRY2zBvdN3703dyX83Ie6BlvPT/Alv'
    'DvNzPMZu4gdNCNwy60Lw56KubblTNZ+IOFtjmTXWFThBd9dH2ZaEzmfClrdU6IUd3UShRXQ8mrBP6Ae5WtMFHecHF7iLWyVwG8jyiqn+B1BLAwQUAAAACAAA'
    'ACFc85t93wQKAABeGgAAEwAAAFNUUjAxX00wMl9sZXNzb24ubWTtWN9PG9kZffdfcaWobw7GSVq1SFUUdVdV1Ga7YvvUqgIWnBUSgQiTXbUPlYEBOZhsDLHB'
    'Zme8dhcS2HXUAQwY1agSf8rmbe6d/6Hn+747nnFgk31s1ZWQsMd37nw/zjnfuXNDffLH0eGsejB8S31fqCh9WtDrBX3sKOP18KfM6rr+dt+4+8o0q8GZG9ax'
    'yH+uTKMdHDeVPi7o53Wln/nG64SOj1WOOarpl22F2/SLogpO2mHNSaVu3FBmeTGsHwTHPfx2EG64qVTgF3BR/e7j+3iyYxxP6VJHl85No6xbrnHcZED8xMsz'
    'iejyTPZQH/zhE1MvqODID6vP8O9CH3f0Xs/uF1YPdKmoS7tD70jNeE6cXVitUWI/kB/tjKWm6WKdXqtcyZf2kpRpwYGfxjffFGsqXDunf6ZWNPUu3Yk4TasT'
    'Vl2KTurzVVkfdxWWhVtPlT6qGNeh3MLnT1GoVOpmtCKsOrSX/mfXrNSGJIaOMqVds3ZitsuoG4fG+8QV6N+PGi4X1W+nF1T4Zdc4h+qWKbmB77wpVGiPbivw'
    'N5GuMlVHl2pvCtX4ybLniDK7G/r4JK100dHeBVe0UU7bkuqvn1I/sBEWlFBhR4W1ntk5wAJvnavA1U2rcLOINupvV1CX3XC5gCvLbnCG52yvcgpeh/ARdJ+a'
    'lx63yetRH79uK+M0jIe+C4IQnmns0Qr9al2/2DVbHdsnhbaa87rKDpt/rEinFxGKrTu27mB3gKNoTl2G5rkTHDvhzjO6H1+AQxuULrv9SAo2NHNeowXB+Zfm'
    'Fa5V1wQ0ic5GYYRfMcjQDqp4cFTAPggqrL5GBYwHgKye4PmAI0Vhy40a4xOVybwqWpjyZnyBHoTnUbsRA+LXzb0ITzfVnwUlfxmsw+XZABBtrErIn2DxO0sn'
    'DHk7mX4CuGH4Z9hJ4J8sMFj7VnnRwDr38RqWoLfKLL1GJLpcu1JfXANEsGYoAc8ln8rQ7o2o/PxkJrcwk8kvzA9nx6YmFibGJmdyE7P56dnPhh7/VUTs+moK'
    'GZuAw4kiJfpmnbiHb8gQFzYoZ1Mt6qXO5dlv5mYmPiXqEWWji8JEpVe64VpXny5GV8IKwck0PVxWf7r/MUd+UdEv3BE1/p6Ax4dA+xNTauPfPmpESiRCg92b'
    'BaA/6svRd1E8hLl790lCzClViuQMCjkgCf2mj6iB8MYpgszoh/c+ePDh0KOpcdbU/R4eBehfIBm7hTDvV4HvkiwhFfS4UjUrFe0UIYvZ7MAvEIdnaFNjBVDN'
    '/gI/IXgJDbGiAb/EJZsZBHxQu0ouaq/M1oYpeagXD64xDK6xh3Pzj/IUIzd1am4yn5nPPZ6bX8gTr0gl6w73TFoXlgAZ6TEwelS0OsoclApHM+umCtfXITk/'
    'oju0lvuCtbOIZ2Jm+m+5scm52fyTR48XpudmxxlCjqlejFi1Ug/nJx7l0l9Mz07NffHr7B31/eqm+nQin7s8m59YyPGo4dqA06wES21gPBpkILKIJwaaXmqa'
    'V64putx82TCecCTWplonUXd7rBTxrLMSFY0F0QQrd9Rwq2QMrKDrm93CMLH1o4mPqHZ871LbShsrBq8R7vNA4sECZPVoStKzX/iWySBvBEG/hvSS8rPscaME'
    '6DxlhYzMFUKPnenRdQhxsYaKMMw4Fo8JYvyuacEYdNSDLPX3c7QFvcuNzeSmPsvNxxwaQNoACkS4iVPOIT+LqcWT5u3iQEcHeKZIDpuM52gW01RyaEfSwZNy'
    'wp5QsVDAe3BC6wX0BmQ57hqvCxHuBYc+iWy4Svantqv3arQh0uQp5Da5VUv7wflilAShO6zWSYXCugsLQ1du2M9ZVnuRYR42S21EBayXMa1TqWj2sL+6PENm'
    'LK1cezyfBZFz8i4oc0ELa3WjSM2SGcmzVIa/hMzFFrHX/gE4bNFImntc1s9Rk3IzOF+/SwsllkR5IGPbZQz3VtDtWmsUxUL654JX6H+9ShXD9vqoasdlorOJ'
    'EojbtdrImdmewZMyhO1VzpM7HvX5yEdmSW7gzs1duxXuRU1jD0gxR0MzNqJJcIyo0Z+zMaZ9fN8K+21xceoWFUj7leDflcgLoJjG32dbODDI7wwHhx3COHhu'
    'Y2OQXsEZd/cKfvVGuQ/tWsTm2F9QvhVfxENmzXVVvS3AEsACxuAeF7jRlq2BLQvbuLgDoytamIj3h7MkhMXgo9GCHCJvUBZvcJcIZZxdWEzARBptnGa47N3G'
    'RpGJI6REUOLnM3g5/qg82LaOkpVO+ERAn0jblCDu+mrc4WqYExfuyQJa2GY9Sp9p4u+tdhLvGR9WKPq8u0YfEt0hCpy6yT7DXsGH61IbwpXmGddt4SI+truE'
    'LZmGFBEAEnTQJlQa7CEwLg8M4c5gEoyda9Smg/MW5ELMoPkGZyz0ugxCiYsczf59NDuc9OxA8naPVzasvXRILslwx3lQVTpFe2KIZwGZcUqxFFk0yxiz9S/W'
    'oJOycVvUeYJyYmBIFePjAn1C0lY5cOyggw+V9TxynLHVYT747LfowIXIaLbF05VOBclpGI1BQLfbsqaH2eV1GQh8biSCYHP9sh8juWfKITtM1mnQZi+1I5vt'
    'dRiXdOgEcVYKvPErBwlGSkH7sCzTMa7e55McsIhG4CKxEyBsxJZ7k9H9lmpEjtiekUAFVNwClmzSj7FH49aT2TX9n8fkOvk3WjX3ZOHxE6yCsctMPRyazH9O'
    'l9niWciCMunEuDQ7FXP0Gk3xNoCGNGO1U8T3pNSkI0sMdCF9wBQG52WPWxmP4+RJQugjKZLHEuHnbp06/AJC+o//gDPxipruHELuLU+sENKmwjBukcQt0zzB'
    'NTwChed5iHnbSr64AAoYm1x05joT1Y5PzGgIEWTqClxTKetymbws+jUrTZSQhXzUxOwQvwQSVzDy3/22hMYqnW53yMFaYsX9ENeB1sLW7ByQ2ui1PT4+NMuE'
    '3mvtOYkVVIjP0Nanc6I8OvpDgAwZrWTDJF2+PMPodx3TcCLtkxJyatLKraIdRqlbtsic8nvPqNSva4NFXZNzU8Kltzc/nSv654o3BXKA5vTAtA4j9B19JyZe'
    'tABIsWIADmVEEHCLja3fNDIJiOT3uXxeYMK1kyFNT2BC25PJdhlnGVNbTLT8tm05Q2Pk/+itjJW6PiutasJTYfaIPPPLM9CIg5bwWbwzot39o5YdR3FLfrIU'
    '//OWgkc39WkvExtylE+GJjHTHnjFetNhLzocxOS6I+SS0fhePU2r91qQtLrqP9LiPjJXXziRGgj92ZggRh61yBhhs0u3BzHMh/MafuZE5cxOiN324Sr6jLlL'
    'Jh1+HfdGZh0f2azThegR8kC2AHgGjRiaBlXALWwUrRuIK5RKvPJ/xxv/jGwoL/777/2H7GsNOaoiW3kzopc6EBy5SGoqbwTWcCbHuCgkXON/AFBLAQIUAxQA'
    'AAAIAAAAIVxuE3ryjwgAAIAUAAASAAAAAAAAAAAAAACkAQAAAABTVFIwMV9NMDJfZm9ybXMubWRQSwECFAMUAAAACAAAACFc85t93wQKAABeGgAAEwAAAAAA'
    'AAAAAAAApAG/CAAAU1RSMDFfTTAyX2xlc3Nvbi5tZFBLBQYAAAAAAgACAIEAAAD0EgAAAAA='
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 미션 핵심 개념과 원본 실습 설명

---

## 🎯 미션 02. 부대별 일일 소모율 정규화 및 재고 변동 데이터 전처리 모듈 구현

**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 부대별 일일 소모율 정규화 및 재고 변동 데이터 전처리 모듈을 구현한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **부대별 일일 소모율을 정규화하고 재고 변동 데이터를 정제하는 전처리 모듈을 구현하며, 군수 특수 예외는 보존한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **중복, 누락 일자, 소모량 결측, 단위 혼용, 음수 재고, 품목명 오타, 태그 없는 이상 급증을 일부러 섞은 가상 원장을 만들어 정제 규칙 10종을 적용 수행하고 검증하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/etl/str01_data_cleansing.py 및 정제 품질 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 특수 예외(비계획 정비, 긴급 불출 등) 데이터 보존의 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 규칙 10종 적용, 데이터 정합성 검증 통과율 100%

- 진행 기준: ETL 파이프라인 구현 및 정제된 마스터 데이터셋 구축

- 검증 방법: 전후 데이터 통계량(기술통계) 비교 및 무결성 검증

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


### AI 활용 예시 · 미션 02 전용

> 나는 Python을 처음 배우는 학습자야. **부대별 일일 소모율 정규화 및 재고 변동 데이터 전처리 모듈 구현**을 진행하고 있어. 다음 요구사항을 보고 핵심 함수의 입력·출력, 필요한 검증 3개를 쉬운 말로 설명해줘. `TODO` 구간만 구현할 수 있는 코드를 제안하고, 데이터를 미리 보지 않는 검증 방법과 경계 사례도 알려줘. 실행 결과는 만들지 말고 내가 실제 실행한 로그를 받은 다음 분석해줘.  
> 요구사항: src/etl/str01_data_cleansing.py / normalize_consumption: 원장 frame,window=14 → base·rate를 추가한 복사본. 부대·품목별 날짜순 첫 window일 소모 중앙값으로 정규화한다. 원본·예외 태그를 보존하고 기준0은 NaN이다. 사후 정제 기준이며 실시간 피처로 쓰지 않는다.

**AI 답변을 받으면:** 그대로 복사하지 말고 입력 시점·자료 범위·실제 실행 여부를 직접 확인하세요. 확인되지 않은 수치는 `미측정`, 수행하지 않은 활동은 `미실시`로 남깁니다.


In [ ]:
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """중복·결측·단위 혼용·음수 재고·품목명 오타 등이 섞인 가상 일일 재고 원장을 10종 정제 규칙으로 정제하고 일소모율을 정규화하되, 긴급 불출과 비계획 정비 같은 군수 특수 예외는 삭제하지 않고 보존하며 정합성 검증 통과율을 보고한다."""
    import numpy as np
    import pandas as pd

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 부대 20개 x 핵심 보급품 5종 일일 재고·소모 시뮬레이터 (실제 부대·수량·기준일수 아님)
    # 핵심성과지표(KPI) = 기준 보급일수(DOS) = 현재 재고 / 최근 평균 일일 소모량. 목표 DOS는 교육용 가정값이다.
    # 결핍 위험 일수 H = 보급이 없다고 가정할 때 현재 재고가 바닥나기까지 남은 일수(미래 실제 소모량으로 계산한 정답).
    # ---------------------------------------------------------------------------
    ITEMS = {"식량": (100, 0.3, 10), "피복": (20, 0.1, 15), "유류": (300, 1.2, 7), "탄약": (60, 1.5, 6), "수리부속": (15, 0.5, 12)}   # (기본 일소모, 템포 민감도, 목표 DOS)
    N_UNIT, DAYS = 20, 160
    UNITS = [f"가상{u:02d}부대" for u in range(1, N_UNIT + 1)]


    def make_data(seed=1, n_unit=N_UNIT, days=DAYS):
        """일 단위 재고 원장: unit, item, day, tempo(실제 작전 템포), plan(사전 계획 템포), wx(악천후), consume, resupply, stock."""
        rng = np.random.default_rng(seed)
        rows = []
        for u in range(1, n_unit + 1):
            s, ph = rng.uniform(0.7, 1.5), rng.uniform(0, 6.28)
            surge = np.zeros(days)
            for st in rng.choice(days - 10, 4, replace=False):
                surge[st:st + int(rng.integers(5, 10))] = rng.uniform(0.4, 0.8)
            tempo = 1 + 0.25 * np.sin(2 * np.pi * np.arange(days) / 30 + ph) + surge
            plan = tempo + rng.normal(0, 0.08, days)
            wx = (rng.random(days) < 0.15).astype(int)
            for it, (b, sens, tg) in ITEMS.items():
                stock, off = tg * b * s * rng.uniform(0.8, 1.2), int(rng.integers(0, 4))
                for t in range(days):
                    c = b * s * (1 + sens * (tempo[t] - 1)) * (1 + 0.12 * wx[t]) * rng.normal(1, 0.08)
                    r = max(tg * b * s - stock, 0) * rng.uniform(0.9, 1.1) * (0.5 if rng.random() < 0.08 else 1) if (t % 4 == off and t > 0) else 0.0   # 4일 주기 정기 보급: 목표 재고까지 채움(가끔 수송 차질로 절반)
                    stock = max(stock + r - c, 0.0)
                    rows.append((f"가상{u:02d}부대", it, t, tempo[t], plan[t], wx[t], c, r, stock))
        return pd.DataFrame(rows, columns=["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"])


    def make_samples(d, lo=21, hi=125):
        """(부대, 품목, 기준일 t0)별 특성(피처)과 정답 H. 피처는 t0 시점까지 알 수 있는 값과 사전 계획·기상 예보만 사용한다."""
        out = []
        for (u, it), g in d.groupby(["unit", "item"]):
            c, stk, rs = g["consume"].values, g["stock"].values, g["resupply"].values
            pl, wx, tp = g["plan"].values, g["wx"].values, g["tempo"].values
            cs = np.cumsum(c)
            tg = ITEMS[it][2] * ITEMS[it][0]
            for t0 in range(lo, hi):
                idx = np.searchsorted(cs, cs[t0] + stk[t0])
                if idx >= len(cs) or idx <= t0:
                    continue
                c3, c7, c14 = c[t0 - 2:t0 + 1].mean(), c[t0 - 6:t0 + 1].mean(), c[t0 - 13:t0 + 1].mean()
                last = [k for k in range(t0, t0 - 15, -1) if rs[k] > 0]
                out.append({"unit": u, "item": it, "t0": t0, "H": idx - t0, "stock": stk[t0], "c3": c3, "c7": c7, "c14": c14,
                            "dos7": stk[t0] / c7, "dos3": stk[t0] / c3, "trend": c7 / c14, "burst": c3 / c7,
                            "plan7": pl[t0 + 1:t0 + 8].mean(), "plan_ratio": pl[t0 + 1:t0 + 8].mean() / tp[t0], "tempo0": tp[t0],
                            "wx_fc": wx[t0 + 1:t0 + 8].mean(), "since_rs": (t0 - last[0]) if last else 15,
                            "rs_avg": rs[t0 - 13:t0 + 1].sum() / 14, "fill": stk[t0] / tg, "item_code": list(ITEMS).index(it)})
        return pd.DataFrame(out)


    def mape(a, b):
        # [TODO : 학생 실습 2] 부대별 일일 소모율 정규화 및 재고 변동 데이터 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    clean_truth = make_data()
    rng = np.random.default_rng(2)
    raw = clean_truth.copy()
    raw["tag"] = ""
    n = len(raw)
    inj = {}
    perm = iter(rng.permutation(n))                      # 오염 종류끼리 같은 행이 겹치지 않도록 한 번에 뽑는다
    def pick(k):
        # [TODO : 학생 실습 2] 부대별 일일 소모율 정규화 및 재고 변동 데이터 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    i_ex = pick(40); raw.loc[i_ex, "consume"] = raw.loc[i_ex, "consume"] * 2.5; raw.loc[i_ex[:20], "tag"] = "긴급불출"; raw.loc[i_ex[20:], "tag"] = "비계획정비"; inj["특수 예외(보존 대상)"] = 40
    i_unit = pick(30); raw.loc[i_unit, "consume"] = raw.loc[i_unit, "consume"] * 1000; inj["단위 혼용(1000배)"] = 30
    i_out = pick(30); raw.loc[i_out, "consume"] = raw.loc[i_out, "consume"] * 6; inj["태그 없는 이상 급증"] = 30
    i_na = pick(200); raw.loc[i_na, "consume"] = np.nan; inj["소모량 결측"] = 200
    i_neg = pick(20); raw.loc[i_neg, "stock"] = -raw.loc[i_neg, "stock"] - 1; inj["음수 재고"] = 20
    i_typo = pick(60); raw.loc[i_typo, "item"] = raw.loc[i_typo, "item"].map({"식량": "식 량", "피복": "피복류", "유류": "유 류", "탄약": "탄 약", "수리부속": "수리 부속"}); inj["품목명 오타"] = 60
    dup = raw.sample(100, random_state=2); inj["중복 행"] = 100
    miss = pick(120)
    gap_rows = raw.iloc[miss]; inj["누락 일자 행"] = 120
    raw = pd.concat([raw.drop(raw.index[miss]), dup]).sample(frac=1, random_state=3).reset_index(drop=True)
    print(f"[정제 전] {len(raw)}행(원래 {len(clean_truth)}행), 주입 오염: {inj} [교육용가상자료]")

    # --- 정제 규칙 10종 ---
    log = {}
    df = raw.copy()
    R = []
    b = len(df); df = df.drop_duplicates(["unit", "item", "day"]); log["R1 중복 행 제거"] = b - len(df)
    FIX = {"식 량": "식량", "피복류": "피복", "유 류": "유류", "탄 약": "탄약", "수리 부속": "수리부속"}
    log["R2 품목명 표준화"] = int(df["item"].isin(FIX).sum()); df["item"] = df["item"].replace(FIX)
    full = pd.MultiIndex.from_product([UNITS, list(ITEMS), range(DAYS)], names=["unit", "item", "day"])
    log["R3 누락 일자 보충"] = len(full) - len(df)
    df = df.set_index(["unit", "item", "day"]).reindex(full).reset_index(); df["tag"] = df["tag"].fillna("")
    for c in ("tempo", "plan", "wx", "resupply"):
        df[c] = df.groupby(["unit", "item"])[c].transform(lambda s: s.interpolate(limit_direction="both"))
    df["wx"] = df["wx"].round().astype(int)
    med = df.groupby(["unit", "item"])["consume"].transform("median")
    unit_err = (df["consume"] > 300 * med) & (df["tag"] == "")
    log["R4 단위 혼용 보정(1000배 -> /1000)"] = int(unit_err.sum()); df.loc[unit_err, "consume"] /= 1000
    out = (df["consume"] > 4 * med) & (df["tag"] == "")
    log["R5 태그 없는 이상 급증 상한 처리(중앙값 x3)"] = int(out.sum()); df.loc[out, "consume"] = 3 * med[out]
    log["R6 특수 예외 보존(긴급불출·비계획정비)"] = int((df["tag"] != "").sum())
    log["R7 소모량 결측 선형 보간"] = int(df["consume"].isna().sum())
    df["consume"] = df.groupby(["unit", "item"])["consume"].transform(lambda s: s.interpolate(limit_direction="both"))
    log["R8 음수 재고 -> 흐름 재계산 대상"] = int((df["stock"] < 0).sum())
    s0 = clean_truth.groupby(["unit", "item"])["stock"].first()
    stk = []
    for (u, it), g in df.groupby(["unit", "item"], sort=False):             # R9 재고 흐름 일관성: stock = max(전일 + 보급 - 소모, 0)
        c, r, cur = g["consume"].values, g["resupply"].values, s0[(u, it)]
        for t in range(len(g)):
            if t > 0:
                cur = max(cur + r[t] - c[t], 0.0)
            else:                                                         # 첫날은 보고 재고를 쓰되 결측·음수면 초기 재고 사용
                first = g["stock"].values[0]
                cur = first if first >= 0 else s0[(u, it)]
            stk.append(cur)
    log["R9 재고 흐름 재계산"] = len(df)
    df["stock"] = stk
    df["base"] = df.groupby(["unit", "item"])["consume"].transform(lambda s: s.iloc[:14].median())
    df["rate"] = df["consume"] / df["base"]                               # R10 일소모율 정규화(부대·품목별 기준 소모량 대비)
    log["R10 일소모율 정규화"] = len(df)
    print("\n[정제 규칙 10종 적용 건수]")
    for k, v in log.items():
        print(f"  {k}: {v}")

    # --- 정합성 검증(정제 후) ---
    chk = {"중복 키 없음": not df.duplicated(["unit", "item", "day"]).any(), "품목명 표준": df["item"].isin(ITEMS).all(),
           "일자 연속(160일)": df.groupby(["unit", "item"])["day"].count().eq(DAYS).all(), "결측 없음": not df[["consume", "stock", "resupply", "tempo"]].isna().any().any(),
           "소모량 0 이상": (df["consume"] >= 0).all(), "재고 0 이상": (df["stock"] >= 0).all(),
           "소모율 합리 범위(0.2~6)": df["rate"].between(0.2, 6.0).all(),
           "특수 예외 태그 보존": (df["tag"] != "").sum() == 40, "단위 이상 없음(1000배 급증 0건)": not (df["consume"] > 50 * df["base"]).any(),
           "키 조합 100개": df.groupby(["unit", "item"]).ngroups == 100}
    for k, v in chk.items():
        print(f"  검증 {k}: {'통과' if v else '실패'}")
    rate = sum(chk.values()) / len(chk)
    print(f"정합성 검증 통과율 {rate:.0%} ({sum(chk.values())}/{len(chk)})")
    cmp = df.merge(clean_truth, on=["unit", "item", "day"], suffixes=("", "_t"))
    bad_rows = cmp[(cmp["tag"] == "") ]
    print(f"복원 품질: 일반 행의 소모량 평균 절대 오차 {float((bad_rows['consume'] - bad_rows['consume_t']).abs().mean()):.3f} (평균 소모량 대비 {float((bad_rows['consume'] - bad_rows['consume_t']).abs().mean() / bad_rows['consume_t'].mean()):.2%})")
    ex = cmp[cmp["tag"] != ""]
    print(f"특수 예외 {len(ex)}건은 값을 그대로 보존(지우거나 깎지 않음): 소모 급증 평균 {float((ex['consume'] / ex['base']).mean()):.1f}배 유지")
    print(f"\n목표(정제 규칙 10종 적용, 정합성 검증 통과율 100%): {'달성' if len(log) >= 10 and rate == 1 else '미달'}")


## 📋 3. 학습활동 4단계

### 활동 1: 무엇을 확인하고 계획하나요?

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 원본 목표, 기본 코드, 그리고 제공된 학습 자료를 면밀히 분석하여, 데이터 내에 존재하는 태그 기반 급증(Spike)이나 단위 오류를 단순한 일괄 방식(예: 동일 비율 적용 등)으로 일괄 수정해도 무방한지 엄밀하게 판단합니다.
  - 아울러 전처리 과정에서 AI 에이전트에 위임할 분석 범위, 데이터 제약 사항, 예외 보존 원칙, 그리고 최종 검증 방법을 체계적으로 수립합니다.
- **수행 방식:**
  - 막연한 코딩이나 AI의 무분별한 수정에 의존하지 않고, 군수 데이터의 특수성(긴급 불출, 비계획 정비 등)을 훼손하지 않기 위한 명확한 사전 규칙과 검증 계획을 문서화합니다.

#### 2. 세부 수행 절차

- **원본 목표·기본 코드·자료 분석 및 태그 급증·단위 오류 검토:**
  - 부대별 일일 소모율 정규화 및 재고 변동 데이터 정제에 필요한 입력 데이터, 기간, 단위, 그리고 사용 가능한 시점의 제약 조건을 파악합니다.
  - 데이터 내 급증 구간이나 단위 오류를 일괄적인 규칙으로 단순 치환할 경우 실제 작전 상황의 특이 변동이 왜곡될 수 있으므로, 예외 처리 기준을 사전에 정의합니다.
- **normalize_consumption 입출력 계약 및 전처리 범위 정의:**
  - 향후 구현할 핵심 함수인 normalize_consumption 의 입력(원장 frame, window=14)과 출력(base rate 추가 복사본) 계약 조건을 이해하고 설계합니다.
- **AI 위임 범위 및 검증 계획 정의:**
  - AI 에이전트에 맡길 전처리 코드 구현 및 정제 보고서 작성의 범위를 설정하고, 입력 데이터의 무결성 및 특수 예외 보존 여부를 검증할 구체적인 확인 방법을 문서( forms/STR01_M02_forms.md )에 명시합니다.

#### 3. 주의 및 검증 사항

- 입력 시점이나 기간, 분모가 서로 다른 수치를 동일한 성능이나 조건으로 잘못 해석하지…

#### 완료 확인:

- 이 활동의 확인 근거 M02 계획: 부대별 일일 소모율을 정규화하고 재고 변동 데이터를 정제하는 전처리 모듈을 구현하며, 군수 특수 예외는 보존한다.에 필요한 입력·기간·단위·사용 가능 시점과 normalize_consumption의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 2: AI와 코드 초안을 만들어 실행하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 핵심 전처리 함수인 normalize_consumption 을 구현·저장하고 첫 성공 실행 결과를 온전히 보존합니다.
  - 부대별·품목별 날짜순 첫 window 일(14일) 소모 중앙값으로 소모율을 정규화하며, 원본 및 예외 태그를 보존하고 기준 값이 0인 경우 NaN 으로 처리합니다.
  - 아울러 사후 정제 기준과 실시간 피처의 차이를 이해하고, 변경 과제(R5 상한 배수를 3에서 2로 변경)를 수행하여 복원 오차와 특수 예외 40건의 값 보존 여부를 비교 검증합니다.
- **수행 방식:**
  - AI 에이전트와 협업하여 요구되는 데이터 전처리 계약 조건을 코드로 구현하고, 변경 전후의 실행 로그와 정제 보고서를 각각 정확하게 확보합니다.

#### 2. 세부 수행 절차

+ **코드 구현 및 첫 실행**
  - 핵심 함수 normalize_consumption 구현 및 첫 성공 실행: 원장 frame 과 window=14 를 입력받아 base_rate 컬럼이 추가된 복사본을 반환하는 함수를 구현합니다.
  - 부대별·품목별 날짜 순서에 따라 초기 window 일 동안의 소모 중앙값을 기준으로 정규화를 수행하며, 원본 및 예외 태그를 손실 없이 유지하고, 기준이 0인 데이터는 NaN 으로 처리합니다.
  - 해당 로직이 사후 정제 기준임을 명확히 인지하고 실시간 예측 피처로 혼용하지 않도록 계약 조건을 준수한 뒤, 첫 성공 실행 결과와 로그를 보존합니다.

+ **변경 과제와 전후 비교**
  - 변경 과제 수행 (상한 배수 3 -> 2 조정): R5 규칙의 상한 배수 조건을 기존 3에서 2로 변경하여 코드를 수정하고 재실행합니다.
  - 상한 배수 변경에 따른 복원 오차의 변화와 군수 특수 예외 40건의 값이 안전하게 보존되는지 비교 분석합니다.

+ **실행 결과와 보고서 보존**
  - **전후 정제 보고서 분리 저장 및 실제 출력 보존:**
    - 기본 실행 결과와 상한 배수 변경 실행 결과의 정제 보고서를 각각 별도의 파일로 나누어 저장하고, 실제 코…

#### 완료 확인:

- 이 활동의 확인 근거 M02 구현: src/etl/str01_data_cleansing.py의 normalize_consumption를저장하고계약〔원장 frame,window=14 → base·rate를 추가한 복사본.
- 부대·품목별 날짜순 첫 window일 소모 중앙값으로 정규화한다.
- 원본·예외 태그를 보존하고 기준0은 NaN이다.
- 사후 정제 기준이며 실시간 피처로 쓰지 않는다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 3: 원자료와 결과를 대조·수정하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 앞서 구현 및 수정된 전처리 코드를 저장하고 재실행하여, 10종의 정합성 검사 규칙 통과율 100% 여부를 확인합니다.
  - 특히 긴급 불출이나 비계획 정비 같은 군수 특수 예외 40건의 값과 태그가 유실 없이 모두 보존되었는지 검증하고, AI의 설명 및 코드 선택 3건 이상을 실제 군수 도메인 자료 및 실행 결과와 철저히 대조하여 보완 답변을 도출합니다.
- **수행 방식:**
  - 단편적인 코드 실행에 그치지 않고, 가상 오염 주입을 전제로 한 정제 규칙 10종의 정합성을 엄정하게 평가하며 AI의 답변을 비판적으로 교차 검증합니다.

#### 2. 세부 수행 절차

- **수정 코드 저장 및 재실행:**
  - 상한 배수 변경 및 전처리 로직이 반영된 코드를 안전하게 저장하고 재실행하여 에러 여부와 실행 로그를 확보합니다.
  - 특수 예외(40건) 값 및 태그 보존 여부 검증: 긴급 불출, 비계획 정비 등의 군수 특수 예외 데이터 40건이 정제 과정에서 임의로 삭제되거나 왜곡되지 않고, 원본 값과 태그가 완벽하게 유지되었는지 결과 데이터프레임과 대조합니다.
  - AI 설명·코드 선택 3건 이상 근거 대조 및 보완: AI가 제안하거나 설명한 코드 선택 사항 중 핵심적인 3건 이상을 선정하여, 실제 군수 실무 자료와 실행 결과 데이터에 비추어 타당한지 교차 검증하고 보완된 답변을 기록합니다.

#### 3. 주의 및 검증 사항

- 규칙 R1～R10은 가상 오염 주입을 전제로 하므로, 기초 재고는 제공된 정답 자료의 교육 설정을 따르며 원인 확인 없이 실제 원장을 임의로 고치지 않도록 주의해야 합니다.
- 보간 및 전체 기간 중앙값은 사후 정제 목적이므로 실시간 예측 입력의 인과 처리와 혼동해서는 안 되며, 정합성 100%는 10개 검사 항목의 통과를 의미할 뿐 모든 셀의 진실 복원 100%를 뜻하는 것이 아님을 인지해야 합니다.

#### 완료 확인:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M02 검증: [필수] 정제 규칙·특수 예외 보존 — 데이터 정제 규칙 10종을 적용하고 정합성 검증 통과율 100%이며, 긴급 불출·비계획 정비 같은 군수 특수 예외가 삭제되지 않고 보존되는가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - **수행 조건과 판정 기준 · 판정 상태와 적용 범위**
    - 규칙 R1～R10은 가상 오염 주입을 전제로 합니다.
    - 기초 재고는 제공된 정답 자료에서 얻는 교육 설정입니다.
    - 실제 원장을 원인 확인 없이 고치지 않습니다.
  - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
    - 보간·전체기간 중앙값은 사후 정제이며 예측 입력의 인과 처리와 다릅니다.
    - 100%는 10개 정합성 검사 통과이고 모든 셀의 진실 복원 100%가 아닙니다.
    - 태그 건수와 원래 예외값도 비교합니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 …

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 4: 최종 결과와 수행 상태 기록하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 미션 2(부대별 일일 소모율 정규화 및 재고 변동 데이터 전처리 모듈 구현) 전 과정을 거치며 산출된 원본 전처리 코드 , 정제 분석 보고서 , 처리 결과 데이터(DataFrame) , 그리고 실습 기록 서식 을 누락 없이 집대성합니다.
  - 아울러 정량적 목표 달성 여부, 미측정 영역, 문서 작성 상태 및 교관(또는 시스템) 승인 대기 상태를 명확히 구분하여 최종 판단을 기록합니다.
- **수행 방식:**
  - 단편적인 파일 제출에 치중하지 않고, 이번 미션에서 수행한 데이터 전처리 과정과 군수 특수 예외 보존 성과를 객관적으로 검수하여 후속 미션(미션 3: 단순 잔여 DOS 계산 및 Baseline 알고리즘 구현)으로 온전히 인계할 수 있도록 패키지를 정비합니다.

#### 2. 세부 수행 절차

- **핵심 산출 파일 패키지 확정 및 제출:**
  - 전처리 모듈 소스 코드인 src/etl/str01_data_cleansing.py , 정제 결과 분석 보고서인 reports/str01_cleansing_report.md , 처리 완료된 데이터 파일인 outputs/M02/df.csv , 그리고 실습 기록 서식인 forms/STR01_M02_forms.md 의 실제 작성 파일들을 최종 점검하고 제출 상태로 정리합니다.
- **코드 변경 전후 및 AI 대화 기록 연계 보존:**
  - 상한 배수 변경(3 → 2) 전후의 코드 실행 결과와, 전처리 로직 구현 과정에서 AI 에이전트와 나눈 핵심 질의응답 및 코드 선택 근거 대화 내용을 실습 대화 보존 위치에 명확히 연결합니다.
- **상태별 최종 판단 기록 작성:**
  - 정량적 목표 달성 여부, 현장 여건상 미처 측정하지 못한 영역, 문서 작성 완료 상태, 그리고 실제 교관 승인 대기 상태를 투명하게 구분하여 서식( forms/STR01_M02_forms.md )에 상세히 기재합니다.

3.

- 주의…

#### 완료 확인:

- 이 활동의 확인 근거 M02 완료: src/etl/str01_data_cleansing.py, reports/str01_cleansing_report.md, outputs/M02/df.csv, forms/STR01_M02_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:


## 🔍 4. 평가 기준과 확인할 자료

- [정량적 평가] [필수] 정제 규칙·특수 예외 보존 — 데이터 정제 규칙 10종을 적용하고 정합성 검증 통과율 100%이며, 긴급 불출·비계획 정비 같은 군수 특수 예외가 삭제되지 않고 보존되는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.
- **[정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.**
  - **PASS 증거**
    - **수행 조건과 판정 기준 · 판정 상태와 적용 범위**
      - 규칙 R1～R10은 가상 오염 주입을 전제로 합니다.
      - 기초 재고는 제공된 정답 자료에서 얻는 교육 설정입니다.
      - 실제 원장을 원인 확인 없이 고치지 않습니다.
    - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
      - 보간·전체기간 중앙값은 사후 정제이며 예측 입력의 인과 처리와 다릅니다.
      - 100%는 10개 정합성 검사 통과이고 모든 셀의 진실 복원 100%가 아닙니다.
      - 태그 건수와 원래 예외값도 비교합니다.
    - **보완 방법**
      - 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.
- **[진행 추적 및 완료 기준] 정성·검증:**
  - **수행 조건과 판정 기준 · 판정 상태와 적용 범위**
    - 규칙 R1～R10은 가상 오염 주입을 전제로 합니다.
    - 기초 재고는 제공된 정답 자료에서 얻는 교육 설정입니다.
    - 실제 원장을 원인 확인 없이 고치지 않습니다.
  - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
    - 보간·전체기간 중앙값은 사후 정제이며 예측 입력의 인과 처리와 다릅니다.
    - 100%는 10개 정합성 검사 통과이고 모든 셀의 진실 복원 100%가 아닙니다.
    - 태그 건수와 원래 예외값도 비교합니다.
  - **제출 파일 및 완료 기록**
    - 파일: src/etl/str01_data_cleansing.py, reports/str01_cleansing_report.md, outputs/M02/df.csv, forms/STR01_M02_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
    - 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
  - **보완 방법**
    - 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.
**실제 출력 없이 달성으로 표시하지 않습니다.**

In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 2
        DELIVERABLE_PATHS = ['src/etl/str01_data_cleansing.py', 'reports/str01_cleansing_report.md', 'outputs/M02/df.csv', 'forms/STR01_M02_forms.md']
        PROJECT_NAME = 'STR01'
        WORK_ROOT = Path(STR01_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'STR01_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'STR01_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                if state['loaded']:
                    state['dirty'] = True
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# STR01 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ STR01 미션 {MISSION_CODE:02d} 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
